# Preprocess the cleaned reviews, then tokenize

The input is `data/cleaned/digikala-comments-cleaned.csv` only. Cleaning already dropped empty bodies, kept rates in `[0, 5]`, and collapsed line breaks. This notebook does the character pass those counts asked for, then hands that text to the ParsBERT tokenizer.

The same two steps are `python -m src preprocess` and `python -m src tokenize`. The functions live in `src/preprocess.py` and `src/tokenize.py`.

What the character pass does, in order:

1. NFKC, then Arabic letters onto Persian letters (`ك` `ي` `ى` and the small set `ة` `ؤ` `أ` `إ` `ٱ` `ۀ` `ھ`). `آ` stays `آ`.
2. ASCII digits and Arabic-Indic digits onto Persian digits. Latin letters stay, including brand names.
3. One half-space inside a word. A repeated half-space collapses to one. A half-space at an end or beside a space is deleted. Diacritics, tatweel, and the other invisible marks are deleted.
4. A URL or an email becomes a space. A run of three identical characters is capped at two. Whitespace is collapsed again.
5. Emoji stay. A review that is only an emoji is still a review.

Stemming and lemmatization are not in this pass. A lemma such as `رفت#رو` is not the surface word the tokenizer was trained on, and it erases polarity (`نمی‌خرم` versus `خریدم`). The tokenizer below is the WordPiece tokenizer of `HooshvareLab/bert-fa-base-uncased`, applied to the text this pass returns.

In [34]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocess import preprocess_text

examples = [
    "كيفيّة ١٢٣ و battery 128GB",
    "می‌‌شود و عالی 😀",
    "ببین http://example.com/a\nتمام!!!!",
    "😀",
]
for original in examples:
    print(original)
    print(preprocess_text(original))
    print()


كيفيّة ١٢٣ و battery 128GB
کیفیه ۱۲۳ و battery ۱۲۸GB

می‌‌شود و عالی 😀
می‌شود و عالی 😀

ببین http://example.com/a
تمام!!!!
ببین تمام!!

😀
😀



## Write the preprocessed file

This reads the cleaned CSV and writes `data/processed/digikala-comments-preprocessed.csv` with five columns: `id`, `body`, `product_id`, `recommendation_status`, and `rate`. `product_id` stays so a later split can keep every review of one product in the same part. A body that becomes empty, for example a review that was only a URL, is dropped. The printed counts are shares of the loaded rows.

In [41]:
import importlib
import src.cleaning
import src.preprocess

importlib.reload(src.cleaning)
importlib.reload(src.preprocess)
from src.preprocess import preprocess_reviews_file

summary = preprocess_reviews_file(
    ROOT / "data/cleaned/digikala-comments-cleaned.csv",
    ROOT / "data/processed/digikala-comments-preprocessed.csv",
)
summary


PreprocessSummary(rows_before=5258513, rows_after=5258351, rows_changed=394910, dropped_empty_body=162)

In [42]:
import polars as pl
pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_str_lengths(160)

preprocessed_df = pl.read_csv(
    ROOT / "data/processed/digikala-comments-preprocessed.csv",
    )

In [43]:
preprocessed_df

id,body,product_id,recommendation_status,rate
i64,str,i64,str,f64
53672599,"""به درد نمیخوره""",252058,"""not_recommended""",1.0
9897229,"""می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بندی بشه""",252058,"""recommended""",0.0
38074516,"""بسته بندیش خوب بود کاربرد و کیفیتشم خیلی خوبه من برای روغن زدم به مژه و لیفت ابرو خریدم""",3331597,"""recommended""",0.0
18628562,"""به نظرم خوبه فقط یکم ظریفه. از رنگش خوشم اومد من برای صابون ابرو استفاده میکنم راضی ام""",3331329,"""recommended""",0.0
53301258,"""معمولیه اگه واسه خونه رنگ کردن شخصی میخواین اوکیه ولی اگه واسه کار دکلره میخواین نه زیاد ولی خوبه""",3255700,"""recommended""",3.0
53266157,"""قبلا هم استفاده کردم اگه بلد باشین کار کردن باهاش رو برای ی پاکسازی پوست سطحی توی خونه عالیه""",3305270,"""recommended""",5.0
40597171,"""خیلی به کارتون میاد قیمتشم مناسبه""",3480048,"""recommended""",5.0
46198656,"""دقیقا مطابق عکس بود""",7626372,"""recommended""",5.0
46901811,"""برا حالت دادن با اسپری یا پودر عالیه""",821812,"""recommended""",5.0


## Tokenize the preprocessed text

The tokenizer runs only after the character pass. The first call downloads `HooshvareLab/bert-fa-base-uncased`. Each review is cut at 512 tokens, including the tokenizer's own special tokens. The parquet keeps `id`, `product_id`, `recommendation_status`, and `rate` beside `input_ids` and `truncated`, so a product split does not need a join back to the CSV.

In [44]:
import importlib
import src.tokenize

importlib.reload(src.tokenize)
from src.tokenize import encode_bodies, load_tokenizer, tokenize_reviews_file

tokenizer = load_tokenizer("HooshvareLab/bert-fa-base-uncased")
sample = [preprocess_text(text) for text in examples]
ids, truncated = encode_bodies(sample, tokenizer, max_length=512)
for text, row, was_cut in zip(sample, ids, truncated):
    print(text)
    print(tokenizer.convert_ids_to_tokens(row))
    print("truncated:", was_cut)
    print()

token_summary = tokenize_reviews_file(
    ROOT / "data/processed/digikala-comments-preprocessed.csv",
    ROOT / "data/tokenized/digikala-comments-tokenized.parquet",
    tokenizer,
)
token_summary


کیفیه ۱۲۳ و battery ۱۲۸GB
['[CLS]', 'کیفی', '##ه', '۱۲۳', 'و', 'battery', '۱۲۸', '##gb', '[SEP]']
truncated: False

می‌شود و عالی 😀
['[CLS]', 'میشود', 'و', 'عالی', '[UNK]', '[SEP]']
truncated: False

ببین تمام!!
['[CLS]', 'ببین', 'تمام', '!', '!', '[SEP]']
truncated: False

😀
['[CLS]', '[UNK]', '[SEP]']
truncated: False

tokenized: 5258351


TokenizeSummary(rows=5258351, truncated=70, max_length=512)

In [45]:
import polars as pl

In [46]:
pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_str_lengths(160)

preprocessed_df = pl.read_csv(
    ROOT / "data/processed/digikala-comments-preprocessed.csv",
    n_rows=50,
    schema_overrides={
        "id": pl.Int64,
        "body": pl.String,
        "product_id": pl.Int64,
        "recommendation_status": pl.String,
        "rate": pl.Float64,
    },
)
print("preprocessed sample:", preprocessed_df.height, "rows")
preprocessed_df

preprocessed sample: 50 rows


id,body,product_id,recommendation_status,rate
i64,str,i64,str,f64
53672599,"""به درد نمیخوره""",252058,"""not_recommended""",1.0
9897229,"""می‌تونست به عنوان یه کالای فرهنگی بهتر بسته بندی بشه""",252058,"""recommended""",0.0
38074516,"""بسته بندیش خوب بود کاربرد و کیفیتشم خیلی خوبه من برای روغن زدم به مژه و لیفت ابرو خریدم""",3331597,"""recommended""",0.0
18628562,"""به نظرم خوبه فقط یکم ظریفه. از رنگش خوشم اومد من برای صابون ابرو استفاده میکنم راضی ام""",3331329,"""recommended""",0.0
53301258,"""معمولیه اگه واسه خونه رنگ کردن شخصی میخواین اوکیه ولی اگه واسه کار دکلره میخواین نه زیاد ولی خوبه""",3255700,"""recommended""",3.0
53266157,"""قبلا هم استفاده کردم اگه بلد باشین کار کردن باهاش رو برای ی پاکسازی پوست سطحی توی خونه عالیه""",3305270,"""recommended""",5.0
40597171,"""خیلی به کارتون میاد قیمتشم مناسبه""",3480048,"""recommended""",5.0
46198656,"""دقیقا مطابق عکس بود""",7626372,"""recommended""",5.0
46901811,"""برا حالت دادن با اسپری یا پودر عالیه""",821812,"""recommended""",5.0


In [47]:
tokenized_df = pl.read_parquet(
    ROOT / "data/tokenized/digikala-comments-tokenized.parquet",
    n_rows=50,
)
tokenized_df = tokenized_df.with_columns(
    pl.Series(
        "tokens",
        [tokenizer.convert_ids_to_tokens(ids) for ids in tokenized_df.get_column("input_ids").to_list()],
    )
)
print("tokenized sample:", tokenized_df.height, "rows")
tokenized_df.select("id", "product_id", "recommendation_status", "rate", "truncated", "tokens")

tokenized sample: 50 rows


id,product_id,recommendation_status,rate,truncated,tokens
i64,i64,str,f64,bool,list[str]
53672599,252058,"""not_recommended""",1.0,false,"[""[CLS]"", ""به"", … ""[SEP]""]"
9897229,252058,"""recommended""",0.0,false,"[""[CLS]"", ""میتونست"", … ""[SEP]""]"
38074516,3331597,"""recommended""",0.0,false,"[""[CLS]"", ""بسته"", … ""[SEP]""]"
18628562,3331329,"""recommended""",0.0,false,"[""[CLS]"", ""به"", … ""[SEP]""]"
53301258,3255700,"""recommended""",3.0,false,"[""[CLS]"", ""معمولی"", … ""[SEP]""]"
53266157,3305270,"""recommended""",5.0,false,"[""[CLS]"", ""قبلا"", … ""[SEP]""]"
40597171,3480048,"""recommended""",5.0,false,"[""[CLS]"", ""خیلی"", … ""[SEP]""]"
46198656,7626372,"""recommended""",5.0,false,"[""[CLS]"", ""دقیقا"", … ""[SEP]""]"
46901811,821812,"""recommended""",5.0,false,"[""[CLS]"", ""برا"", … ""[SEP]""]"
